# Figure 1: Bayes factors and theoretical QQNM SNR

Input: `data/fig01_bayes_factor_snr.npz`. It contains the evidence values and network-SNR posterior samples used by the submitted paper at commit `8359041`. Time is in seconds; Bayes factors and SNR are dimensionless. The 4 and 5 $M_f$ points use the per-sample median over 64 Gaussian parent-amplitude draws.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', 'Wswiglal-redir-stdio')

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')

fig_width = 3 * 246.0 / 72.27
fig_height = fig_width * (np.sqrt(5) - 1.0) / 2.0
mpl.rcParams.update({
    'axes.labelsize': 24,
    'font.family': 'serif',
    'font.serif': 'Computer Modern Roman',
    'font.size': 24,
    'legend.fontsize': 20,
    'xtick.labelsize': 24,
    'ytick.labelsize': 24,
    'axes.grid': True,
    'text.usetex': True,
    'savefig.dpi': 100,
    'lines.markersize': 14,
    'figure.figsize': [fig_width, fig_height],
})

MTSUN_SI = 4.925490947641267e-6

data = np.load(ROOT / 'data/fig01_bayes_factor_snr.npz')
toffset = data['toffset']
bf_all = np.exp(data['all_logz'] - data['nofourfour_logz'])
bf_removeq = np.exp(data['removeq_logz'] - data['nofourfour_logz'])
snr_samples = [data[f'snr_{label}'] for label in range(4, 11)]

In [ ]:
fig, (ax_bf, ax_snr) = plt.subplots(
    2, 1, sharex=True, figsize=(fig_width, fig_height * 1.40),
    gridspec_kw={'height_ratios': [1.6, 0.72], 'hspace': 0.06})

ax_bf.plot(toffset, bf_all, marker='s', lw=3, color='C0',
           label='linear + quadratic QNMs')
ax_bf.plot(toffset, bf_removeq, marker='o', lw=3, ls='--', color='C2',
           label='remove QQNMs')
ax_bf.axhline(1.0, color='gray', lw=1.5, ls=':', zorder=0)

x4 = float(toffset[1])
y_top, y_bottom = float(bf_all[1]), float(bf_removeq[1])
ax_bf.annotate('', xy=(x4, y_top), xytext=(x4, y_bottom),
               arrowprops=dict(arrowstyle='<->', color='k', lw=2,
                               shrinkA=8, shrinkB=8))
ax_bf.text(x4 + 4e-5, np.sqrt(y_top * y_bottom), 'Bayes Factor = 62',
           fontsize=14, va='center', ha='left',
           bbox=dict(boxstyle='round,pad=0.3', facecolor='lightgray',
                     edgecolor='k', alpha=0.9))
ax_bf.set_ylabel('Bayes Factor\n(v.s. 44 mode removed)', fontsize=19)
ax_bf.set_yscale('log')
ax_bf.set_ylim(0.2, 1e3)
ax_bf.legend(loc='upper right', fontsize=17)
ax_bf.grid(True, alpha=0.3)

positions = toffset[1:]
median = np.array([np.median(samples) for samples in snr_samples[1:]])
lower, upper = np.array([np.percentile(samples, [16, 84]) for samples in snr_samples[1:]]).T
ax_snr.errorbar(positions, median, yerr=[median - lower, upper - median],
                fmt='o', color='C2', ms=9, lw=3, capsize=6, capthick=3, zorder=3)
ax_snr.set_yscale('log')
ax_snr.set_ylim(0.35, 6.5)
ax_snr.set_yticks([0.5, 1, 2, 5])
ax_snr.set_yticklabels(['0.5', '1', '2', '5'])
ax_snr.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
ax_snr.axhline(1.0, color='gray', lw=1.5, ls=':', zorder=0)
ax_snr.set_ylabel('Theoretical\nQQNM SNR', fontsize=19)
ax_snr.set_xlabel('Time After Merger (s)')
ax_snr.grid(True, alpha=0.3)
ax_bf.set_xlim(0.0009, 0.0031)

ax_top = ax_bf.twiny()
conversion_factor = 67.68112439526381 * MTSUN_SI
ax_top.set_xticks(ax_snr.get_xticks())
ax_top.set_xticklabels([f'{tick / conversion_factor:.1f}' for tick in ax_snr.get_xticks()])
ax_top.set_xlim(ax_bf.get_xlim())
ax_top.set_xlabel('Time After Merger ($M_\\mathrm{f}$)')
ax_top.grid(False)
for axis in (ax_bf, ax_snr):
    axis.axvspan(0.0009, 0.00133, alpha=0.2, color='gray')

print(f'4 Mf QQNM BF = {np.exp(data["all_logz"][1] - data["removeq_logz"][1]):.2f}')
for time, center, low, high in zip(positions, median, lower, upper):
    print(f'{time * 1e3:.2f} ms: median {center:.2f}, 68% [{low:.2f}, {high:.2f}]')

output = ROOT / 'figures/bayes_factor_snr_gaussian_bar1sig_arrow_drawmedian.pdf'
fig.savefig(output, bbox_inches='tight')
plt.show()